# M1-D01: Bedrock foundations

What this shows:
1. One Converse API request (messages, system prompt, inference settings, token usage)
2. A bounded tool-use loop: the model **proposes** a tool call, our code **validates and runs** it
3. One error path (missing record)
4. The five-prompt model comparison and the default model choice

In [1]:
import json
from pprint import pprint

from controls_assistant.config import load_settings
from controls_assistant.state import load_state

settings = load_settings()
state = load_state()
print("Profile:", settings.profile, "| Region:", settings.region, "| Default model:", settings.default_model_id)

Profile: vr-training | Region: us-east-1 | Default model: us.amazon.nova-2-lite-v1:0


## 1. One Converse request (no tools)

In [2]:
from controls_assistant.agent_loop import ask, make_client

client = make_client(settings.profile, settings.region)
r = ask(client, settings.default_model_id, "In one sentence, what is a detective control?", use_tools=False)
print(r.answer)
print(f"tokens in={r.input_tokens} out={r.output_tokens} latency={r.latency_ms:.0f} ms stop={r.stop_reason}")

A detective control identifies and reports unauthorized activities or policy violations after they have occurred.
tokens in=117 out=19 latency=1068 ms stop=end_turn


## 2. Tool call

`stopReason == "tool_use"` means the model is asking for a tool. `tools.run_tool()` checks the tool name and arguments, runs it, and sends back a `toolResult`.
Limits live in code: `MAX_TOOL_CALLS = 3`, `TOOL_TIMEOUT_SECONDS = 5`.

In [3]:
r = ask(client, settings.default_model_id, "What is the status of CTL-003 and what was the finding?")
print(r.answer)
print("tool calls:", r.tool_calls)

CTL-003, "Change approval before production deploy", is currently **deficient**. The last test on 2026-10-20 found that **3 of 25 sampled deploys had no recorded approval**. The next test is due on 2026-11-20.
tool calls: [{'name': 'get_control_status', 'input': {'control_id': 'CTL-003'}, 'status': 'success'}]


## 3. Error path: the record does not exist

In [4]:
r = ask(client, settings.default_model_id, "Is CTL-999 effective?")
print(r.answer)
print("tool calls:", r.tool_calls)

Sorry, I can't find any control with the ID CTL-999, so I can't tell you its status. The control ID might be incorrect or not in our system.
tool calls: [{'name': 'get_control_status', 'input': {'control_id': 'CTL-999'}, 'status': 'error'}]


Argument validation happens in code, not in the model:

In [5]:
from controls_assistant.tools import ToolError, run_tool

for args in [{"control_id": "CTL-001"}, {"control_id": "ctl-1"}, {"control_id": "CTL-001", "extra": 1}]:
    try:
        print(args, "->", run_tool("get_control_status", args)["status"])
    except ToolError as e:
        print(args, "-> ToolError:", e)

{'control_id': 'CTL-001'} -> effective
{'control_id': 'ctl-1'} -> ToolError: Invalid control_id 'ctl-1'. Expected a value like CTL-001.
{'control_id': 'CTL-001', 'extra': 1} -> ToolError: Expected exactly one argument: control_id.


## 4. Model comparison (5 prompts x 3 runs)

Full detail: `evidence/m1-d01/README.md`

In [6]:
from pathlib import Path
d = json.loads(Path("../evidence/m1-d01/model_comparison.json").read_text())
print("commit:", d["commit"], "| run at:", d["run_at"])
for model, s in d["summary"].items():
    print(f"{model:35} checks={s['checks_passed']:6} median={s['median_latency_ms']:>6} ms  avg cost=${s['avg_cost_usd']:.5f}")

commit: 44793ff | run at: 2026-11-19T16:15:17+00:00
us.anthropic.claude-sonnet-5        checks=12/15  median=  3195 ms  avg cost=$0.00501
us.amazon.nova-2-lite-v1:0          checks=15/15  median=  1294 ms  avg cost=$0.00077


**Default: Nova 2 Lite.** Same correctness on these prompts, about 2.5x faster and 6.5x cheaper. Sonnet 5 stays as the fallback (one line in `.env`).